<a href="https://colab.research.google.com/github/Lencili/gz-hubble-classifier/blob/main/01-pinn/PINN_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Physics-Informed Neural Networks (PINN) u PyTorch-u

Isti primeri kao u notebooku **17_PINN.ipynb**, samo napisani u **PyTorch-u** umesto u TensorFlow-u.

| Lab | Problem | Kako je urađeno |
|---|---|---|
| 1 | du/dt = cos(2πt), u(0) = 1 | PINN napisan ručno, u čistom PyTorch-u |
| 2 | isti problem kao Lab 1 | DeepXDE biblioteka sa PyTorch backendom |
| 3 | Ojlerova greda, y'''' + 1 = 0 | ručno, u čistom PyTorch-u |
| 4 | Navije–Stoksove jednačine (tok između ploča) | ručno, u čistom PyTorch-u |



## TensorFlow → PyTorch: šta je šta

| Šta radimo | TensorFlow (original) | PyTorch (ovde) |
|---|---|---|
| mreža | `tf.keras.Sequential([Dense(32, 'tanh'), Dense(1)])` | `nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 1))` |
| „prati izvod po t“ | `tape.watch(t)` | `t.requires_grad_(True)` |
| izvod du/dt | `tape.gradient(u, t)` | `torch.autograd.grad(u, t, ..., create_graph=True)` |
| izvod gubitka po težinama | `tape.gradient(loss, NN.trainable_variables)` | `loss.backward()` |
| korak optimizera | `optm.apply_gradients(...)` | `optimizer.step()` (pre toga `optimizer.zero_grad()`) |

**`create_graph=True`** je ekvivalent dve ugnježdene trake u TensorFlow-u: gubitak sadrži izvod du/dt, pa PyTorch mora da zapamti kako je on nastao da bi kasnije mogao da izračuna i izvod gubitka po težinama.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Uređaj:', device)

In [ ]:
def izvod(y, x):
    # Izvod izlaza y po ulazu x, preko automatske diferencijacije.
    # create_graph=True: da bismo mogli da diferenciramo i sam izvod (drugi izvod, trening).
    return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y), create_graph=True)[0]

---
# Lab 1: PINN napisan ručno

**Problem:**

$$\frac{du}{dt} = \cos(2\pi t), \qquad u(0) = 1, \qquad t \in [0, 2]$$

**Tačno rešenje** (služi samo za proveru na kraju):

$$u(t) = \frac{1}{2\pi}\sin(2\pi t) + 1$$

**Gubitak:**

$$\mathcal{L} = \underbrace{\frac{1}{n}\sum_i \left(\frac{dNN(t_i)}{dt} - \cos 2\pi t_i\right)^2}_{\text{da li kriva poštuje pravilo}} + \underbrace{\left(NN(0) - 1\right)^2}_{\text{da li kreće od 1}}$$

### 1.1 Mreža i optimizer

In [ ]:
NN = nn.Sequential(
    nn.Linear(1, 32),   # ulaz: jedan broj t → 32 neurona
    nn.Tanh(),          # glatka aktivacija (ima lepe izvode, za razliku od ReLU)
    nn.Linear(32, 1)    # izlaz: jedan broj u
).to(device)

print(NN)

optm = torch.optim.Adam(NN.parameters(), lr=0.001)

### 1.2 Gubitak (srce PINN-a)

1. Mreža za svako t predloži u.
2. Autodiferencijacija izračuna nagib du/dt.
3. Uporedimo nagib sa pravilom cos(2πt). To je **rezidual**.
4. Proverimo početni uslov u(0) = 1.

In [ ]:
def ode_system(t, net):
    t = t.clone().requires_grad_(True)          # = tape.watch(t)

    u = net(t)                                  # mreža crta krivu
    u_t = izvod(u, t)                           # du/dt (izvod po ULAZU)

    ode_loss = u_t - torch.cos(2 * np.pi * t)   # rezidual: koliko kriva krši pravilo

    t_0 = torch.zeros((1, 1), device=device)
    IC_loss = net(t_0) - 1.0                    # početni uslov: u(0) mora biti 1

    total_loss = torch.mean(ode_loss**2) + torch.mean(IC_loss**2)
    return total_loss

### 1.3 Trening

`train_t` su **kolokacione tačke**: 30 nasumičnih trenutaka između 0 i 2 u kojima proveravamo pravilo. To **nisu merenja**. Mreža nikad ne vidi prave vrednosti u.

In [ ]:
train_t = torch.rand(30, 1, device=device) * 2
train_loss_record = []

for itr in range(6000):
    optm.zero_grad()                       # obriši gradijente iz prethodnog koraka
    train_loss = ode_system(train_t, NN)   # izračunaj gubitak
    train_loss.backward()                  # izvod gubitka po TEŽINAMA (dL/dθ)
    optm.step()                            # popravi težine

    train_loss_record.append(train_loss.item())
    if itr % 1000 == 0:
        print(itr, train_loss.item())

plt.figure(figsize=(6, 4))
plt.plot(train_loss_record)
plt.yscale('log')
plt.xlabel('korak')
plt.ylabel('gubitak')
plt.show()

### 1.4 Rezultat: PINN prema tačnom rešenju

In [ ]:
test_t = np.linspace(0, 2, 100)
train_t_np = train_t.cpu().numpy()

train_u = np.sin(2*np.pi*train_t_np) / (2*np.pi) + 1   # samo za crtanje tačaka na pravoj krivoj
true_u = np.sin(2*np.pi*test_t) / (2*np.pi) + 1

with torch.no_grad():
    pred_u = NN(torch.tensor(test_t, dtype=torch.float32, device=device).reshape(-1, 1)).cpu().numpy().ravel()

plt.figure(figsize=(6, 4))
plt.plot(train_t_np, train_u, 'ok', label='Kolokacione tačke')
plt.plot(test_t, true_u, '-k', label='Tačno')
plt.plot(test_t, pred_u, '--r', label='PINN')
plt.legend(fontsize=12)
plt.xlabel('t', fontsize=15)
plt.ylabel('u', fontsize=15)
plt.show()

print('Najveća greška:', np.max(np.abs(pred_u - true_u)))

**Probaj sama:** promeni jednačinu u onu iz beleški, du/dt = −0,7·u (tačno rešenje e^(−0,7t)). U `ode_system` zameni red sa rezidualom sa:

```python
ode_loss = u_t + 0.7 * u
```

a u ćeliji sa rezultatom stavi `true_u = np.exp(-0.7 * test_t)`.

---
# Lab 2: isti problem preko DeepXDE (PyTorch backend)

DeepXDE sam pravi gubitak i trening petlju. Mi samo opišemo problem.

Backend se bira **pre prvog** `import deepxde`, preko promenljive okruženja `DDE_BACKEND`. Ako si u ovoj sesiji već uvezla DeepXDE sa drugim backendom, uradi Runtime → Restart session pa pokreni ove ćelije ponovo (prvo i ćelije sa importima na vrhu).

| Lab 1 (ručno) | Lab 2 (DeepXDE) |
|---|---|
| `izvod(u, t)` | `dde.grad.jacobian(u, t)` |
| `IC_loss = net(t_0) - 1` | `dde.icbc.IC(geom, lambda t: 1, ...)` |
| `train_t` (30 tačaka) | `num_domain = 30` |
| `nn.Sequential(...)` | `dde.nn.FNN([1, 32, 1], 'tanh', ...)` |
| trening petlja | `model.train(iterations=6000)` |

In [ ]:
!pip install -q deepxde

In [ ]:
import os
os.environ['DDE_BACKEND'] = 'pytorch'

import deepxde as dde
print('DeepXDE backend:', dde.backend.backend_name)

In [ ]:
def ode_system_dde(t, u):
    du_t = dde.grad.jacobian(u, t)
    return du_t - torch.cos(2 * np.pi * t)

def initial(t, on_initial):
    return on_initial and np.isclose(t[0], 0)

def true_solution(t):
    return np.sin(2*np.pi*t) / (2*np.pi) + 1

geom = dde.geometry.TimeDomain(0, 2)
ic = dde.icbc.IC(geom, lambda t: 1, initial)

data = dde.data.PDE(geom,
                    ode_system_dde,
                    ic,
                    num_domain=30,
                    num_boundary=2,
                    solution=true_solution,
                    num_test=100)

net = dde.nn.FNN([1] + [32] + [1], 'tanh', 'Glorot uniform')

model = dde.Model(data, net)
model.compile('adam', lr=0.001, metrics=['l2 relative error'])

In [ ]:
losshistory, train_state = model.train(iterations=6000)
dde.saveplot(losshistory, train_state, issave=False, isplot=True)

---
# Lab 3: Ojlerova greda (savijanje grede)

**Jednačina** (greda dužine 1, pod ravnomernim opterećenjem):

$$\frac{d^4 y}{dx^4} + 1 = 0, \qquad x \in [0, 1]$$

**Granični uslovi:**

| uslov | značenje |
|---|---|
| y(0) = 0 | levi kraj je pričvršćen, ne pomera se |
| y'(0) = 0 | levi kraj je uklješten, ne naginje se |
| y''(1) = 0 | desni kraj je slobodan: nema momenta savijanja |
| y'''(1) = 0 | desni kraj je slobodan: nema sile smicanja |

**Tačno rešenje:**

$$y(x) = -\frac{x^4}{24} + \frac{x^3}{6} - \frac{x^2}{4}$$

Novo u odnosu na Lab 1 su samo **viši izvodi**: funkciju `izvod` primenimo četiri puta zaredom.

Ovo je zapravo **ODE** (zavisi samo od x), iako se u originalu funkcija zove `pde`.

In [ ]:
torch.manual_seed(0)

net_beam = nn.Sequential(
    nn.Linear(1, 20), nn.Tanh(),
    nn.Linear(20, 20), nn.Tanh(),
    nn.Linear(20, 20), nn.Tanh(),
    nn.Linear(20, 1)
).to(device)

optm_beam = torch.optim.Adam(net_beam.parameters(), lr=0.001)

x_domain = torch.rand(10, 1, device=device)   # 10 kolokacionih tačaka unutar grede (kao num_domain=10)
x_left = torch.zeros(1, 1, device=device)     # levi kraj, x = 0
x_right = torch.ones(1, 1, device=device)     # desni kraj, x = 1

def beam_loss(net):
    # 1) jednačina unutar grede
    x = x_domain.clone().requires_grad_(True)
    y = net(x)
    y_x = izvod(y, x)
    y_xx = izvod(y_x, x)
    y_xxx = izvod(y_xx, x)
    y_xxxx = izvod(y_xxx, x)
    pde = y_xxxx + 1

    # 2) levi kraj: y(0) = 0 i y'(0) = 0
    xl = x_left.clone().requires_grad_(True)
    yl = net(xl)
    yl_x = izvod(yl, xl)

    # 3) desni kraj: y''(1) = 0 i y'''(1) = 0
    xr = x_right.clone().requires_grad_(True)
    yr = net(xr)
    yr_xx = izvod(izvod(yr, xr), xr)
    yr_xxx = izvod(yr_xx, xr)

    return (torch.mean(pde**2)
            + torch.mean(yl**2) + torch.mean(yl_x**2)
            + torch.mean(yr_xx**2) + torch.mean(yr_xxx**2))

loss_record_beam = []
for itr in range(5000):
    optm_beam.zero_grad()
    loss = beam_loss(net_beam)
    loss.backward()
    optm_beam.step()
    loss_record_beam.append(loss.item())
    if itr % 1000 == 0:
        print(itr, loss.item())

In [ ]:
x_test = np.linspace(0, 1, 100)
y_true = -(x_test**4)/24 + x_test**3/6 - x_test**2/4

with torch.no_grad():
    y_pred = net_beam(torch.tensor(x_test, dtype=torch.float32, device=device).reshape(-1, 1)).cpu().numpy().ravel()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(loss_record_beam)
ax[0].set_yscale('log')
ax[0].set_xlabel('korak')
ax[0].set_ylabel('gubitak')

ax[1].plot(x_test, y_true, '-k', label='Tačno')
ax[1].plot(x_test, y_pred, '--r', label='PINN')
ax[1].set_xlabel('x (položaj na gredi)')
ax[1].set_ylabel('y (ugib)')
ax[1].legend()
plt.show()

print('Relativna L2 greška:', np.linalg.norm(y_pred - y_true) / np.linalg.norm(y_true))

---
# Lab 4: Navije–Stoksove jednačine (tok između dve ploče)

Tečnost ulazi s leve strane ravnomernom brzinom, lepi se za zidove i postepeno formira **paraboličan profil**: najbrža je u sredini, a uz zidove stoji.

**Mreža:** ulaz (x, y), izlaz (u, v, p):
- u: brzina napred (pravac x)
- v: brzina gore-dole (pravac y)
- p: pritisak

**Tri jednačine** (sve tri treba da budu 0 u svakoj tački):

$$
\begin{aligned}
u\,u_x + v\,u_y + \tfrac{1}{\rho}p_x - \nu(u_{xx} + u_{yy}) &= 0 \quad\text{(kretanje napred)}\\
u\,v_x + v\,v_y + \tfrac{1}{\rho}p_y - \nu(v_{xx} + v_{yy}) &= 0 \quad\text{(kretanje gore-dole)}\\
u_x + v_y &= 0 \quad\text{(ništa ne nestaje)}
\end{aligned}
$$

**Šest graničnih uslova:**

| ivica | uslovi |
|---|---|
| zidovi (y = ±0,5) | u = 0, v = 0 |
| ulaz (x = −1) | u = 1, v = 0 |
| izlaz (x = 1) | p = 0, v = 0 |

### 4.1 Parametri

In [ ]:
rho = 1      # gustina
mu = 1       # viskoznost
u_in = 1     # ulazna brzina
D = 1        # razmak između ploča
L = 2        # dužina kanala
nu = mu / rho

### 4.2 Tačke (kolokacione i granične)

- **2000 tačaka unutra**, gde se proveravaju jednačine. Biramo ih Sobolovim nizom, koji tačke raspoređuje ravnomerno (kao Hammersley u DeepXDE-u).
- **200 tačaka na ivicama**: po 70 na svakom zidu, 30 na ulazu i 30 na izlazu.

In [ ]:
torch.manual_seed(0)

sobol = torch.quasirandom.SobolEngine(dimension=2, scramble=True, seed=0)
s = sobol.draw(2000)
X_domain = torch.stack([s[:, 0] * L - L/2, s[:, 1] * D - D/2], dim=1)

def tacke_na_liniji(n, x=None, y=None):
    # n tačaka na vertikalnoj liniji (zadato x) ili horizontalnoj liniji (zadato y)
    if x is not None:
        yy = torch.rand(n, 1) * D - D/2
        return torch.cat([torch.full((n, 1), float(x)), yy], dim=1)
    xx = torch.rand(n, 1) * L - L/2
    return torch.cat([xx, torch.full((n, 1), float(y))], dim=1)

X_wall = torch.cat([tacke_na_liniji(70, y=-D/2), tacke_na_liniji(70, y=D/2)])   # donji i gornji zid
X_inlet = tacke_na_liniji(30, x=-L/2)                                            # ulaz
X_outlet = tacke_na_liniji(30, x=L/2)                                            # izlaz

X_domain, X_wall, X_inlet, X_outlet = [t.to(device) for t in (X_domain, X_wall, X_inlet, X_outlet)]

plt.figure(figsize=(6, 4))
plt.scatter(X_domain[:, 0].cpu(), X_domain[:, 1].cpu(), s=0.5, label='unutra')
for X, lbl in [(X_wall, 'zidovi'), (X_inlet, 'ulaz'), (X_outlet, 'izlaz')]:
    plt.scatter(X[:, 0].cpu(), X[:, 1].cpu(), s=4, label=lbl)
plt.xlabel('x (dužina kanala)')
plt.ylabel('y (udaljenost od sredine)')
plt.legend(loc='upper right', fontsize=8)
plt.show()

### 4.3 Mreža i fizika

`izvod(u, X)` vraća **oba** prva izvoda odjednom, jer X ima dve kolone: kolona 0 je ∂u/∂x, a kolona 1 je ∂u/∂y. Za drugi izvod ponovo diferenciramo jednu od tih kolona.

In [ ]:
torch.manual_seed(0)

layers = [nn.Linear(2, 64), nn.Tanh()]
for _ in range(4):
    layers += [nn.Linear(64, 64), nn.Tanh()]
layers += [nn.Linear(64, 3)]
net_ns = nn.Sequential(*layers).to(device)   # 2 → 64×5 → 3

def pde(X):
    X = X.clone().requires_grad_(True)
    Y = net_ns(X)
    u, v, p = Y[:, 0:1], Y[:, 1:2], Y[:, 2:3]

    # prvi izvodi: [:, 0:1] je po x, [:, 1:2] je po y
    du = izvod(u, X); du_x, du_y = du[:, 0:1], du[:, 1:2]
    dv = izvod(v, X); dv_x, dv_y = dv[:, 0:1], dv[:, 1:2]
    dp = izvod(p, X); dp_x, dp_y = dp[:, 0:1], dp[:, 1:2]

    # drugi izvodi
    du_xx = izvod(du_x, X)[:, 0:1]
    du_yy = izvod(du_y, X)[:, 1:2]
    dv_xx = izvod(dv_x, X)[:, 0:1]
    dv_yy = izvod(dv_y, X)[:, 1:2]

    pde_u = u*du_x + v*du_y + 1/rho * dp_x - nu*(du_xx + du_yy)
    pde_v = u*dv_x + v*dv_y + 1/rho * dp_y - nu*(dv_xx + dv_yy)
    pde_cont = du_x + dv_y
    return pde_u, pde_v, pde_cont

def mse(a):
    return torch.mean(a**2)

def ukupan_gubitak():
    pde_u, pde_v, pde_cont = pde(X_domain)
    Yw, Yi, Yo = net_ns(X_wall), net_ns(X_inlet), net_ns(X_outlet)

    loss_pde = mse(pde_u) + mse(pde_v) + mse(pde_cont)          # L_G
    loss_bc = (mse(Yw[:, 0]) + mse(Yw[:, 1])                    # zidovi: u = 0, v = 0
               + mse(Yi[:, 0] - u_in) + mse(Yi[:, 1])           # ulaz:   u = 1, v = 0
               + mse(Yo[:, 2]) + mse(Yo[:, 1]))                 # izlaz:  p = 0, v = 0
    return loss_pde + loss_bc, loss_pde, loss_bc                # L_total = L_G + L_BC

### 4.4 Trening (Adam)

Na GPU-u ovo traje par minuta, a na CPU-u znatno duže.

In [ ]:
optm_ns = torch.optim.Adam(net_ns.parameters(), lr=1e-3)
loss_record_ns = []

for itr in range(10000):
    optm_ns.zero_grad()
    loss, loss_pde, loss_bc = ukupan_gubitak()
    loss.backward()
    optm_ns.step()
    loss_record_ns.append(loss.item())
    if itr % 1000 == 0:
        print(f'{itr:6d}  ukupno {loss.item():.4e}  fizika {loss_pde.item():.4e}  ivice {loss_bc.item():.4e}')

plt.figure(figsize=(6, 4))
plt.plot(loss_record_ns)
plt.yscale('log')
plt.xlabel('korak')
plt.ylabel('gubitak')
plt.show()

### 4.5 Rezultati posle Adama (u, v, p odozgo)

In [ ]:
def nacrtaj_polja():
    nx, ny = 200, 100
    xs = np.linspace(-L/2, L/2, nx)
    ys = np.linspace(-D/2, D/2, ny)
    XX, YY = np.meshgrid(xs, ys)
    pts = torch.tensor(np.stack([XX.ravel(), YY.ravel()], axis=1), dtype=torch.float32, device=device)
    with torch.no_grad():
        res = net_ns(pts).cpu().numpy()

    imena = ['u (brzina napred)', 'v (brzina gore-dole)', 'p (pritisak)']
    color_legend = [[0, 1.5], [-0.3, 0.3], [0, 35]]
    for idx in range(3):
        plt.figure(figsize=(8, 3))
        plt.pcolormesh(XX, YY, res[:, idx].reshape(ny, nx), cmap='jet', shading='auto',
                       vmin=color_legend[idx][0], vmax=color_legend[idx][1])
        plt.colorbar()
        plt.title(imena[idx])
        plt.tight_layout()
        plt.show()

nacrtaj_polja()

### 4.6 Dodatni trening (L-BFGS)

L-BFGS je precizniji optimizer za fino doterivanje na kraju. Za razliku od Adama, on više puta zove funkciju gubitka u jednom koraku, pa mu se gubitak predaje kao funkcija (`closure`).

In [ ]:
optm_lbfgs = torch.optim.LBFGS(net_ns.parameters(), lr=1.0, max_iter=3000,
                               tolerance_grad=1e-9, tolerance_change=1e-12,
                               history_size=50, line_search_fn='strong_wolfe')

def closure():
    optm_lbfgs.zero_grad()
    loss, _, _ = ukupan_gubitak()
    loss.backward()
    return loss

optm_lbfgs.step(closure)

loss, loss_pde, loss_bc = ukupan_gubitak()
print(f'Posle L-BFGS: ukupno {loss.item():.4e}  fizika {loss_pde.item():.4e}  ivice {loss_bc.item():.4e}')

### 4.7 Rezultati posle Adam + L-BFGS

In [ ]:
nacrtaj_polja()

### 4.8 Provera: profil brzine na izlazu

Za potpuno razvijen tok između ploča tačan profil je parabola:

$$u(y) = \frac{3}{2}\,u_{in}\left[1 - \left(\frac{y}{D/2}\right)^2\right]$$

U sredini (y = 0) brzina je 1,5, a uz zidove 0.

In [ ]:
y = np.linspace(-0.5, 0.5, 1000).reshape(-1, 1)
x = np.ones_like(y) * (L/2)
outlet = torch.tensor(np.hstack([x, y]), dtype=torch.float32, device=device)

analytic_solution = u_in * 1.5 * (1 - (y / (D/2))**2)
with torch.no_grad():
    PINN_solution = net_ns(outlet).cpu().numpy()

plt.figure(figsize=(6, 4))
plt.plot(y, PINN_solution[:, 0], c='tomato', linewidth=3, label='PINN')
plt.plot(y, analytic_solution, c='k', linestyle='--', label='Tačno (parabola)')
plt.xticks(np.linspace(-0.5, 0.5, 5), fontsize=12)
plt.yticks(np.linspace(0, 1.5, 7), fontsize=12)
plt.legend(fontsize=12)
plt.xlabel('udaljenost od sredine (m)', fontsize=13)
plt.ylabel('brzina u', fontsize=13)
plt.show()

print('Brzina u sredini (PINN):', PINN_solution[500, 0], ' tačno: 1.5')